In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# create a symlink to the hard-coded path in data.yaml
!rm -rf /content/UNIFIED_SPLIT
!ln -s "/content/drive/MyDrive/Dataset" /content/UNIFIED_SPLIT

In [ ]:
# install ultralytics
!pip install -U ultralytics -q

In [ ]:
import os
import shutil
import yaml
import random
import pandas as pd
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

from IPython.display import display, Image
from ultralytics import YOLO

import ultralytics
ultralytics.checks()

In [ ]:
# CONFIG
HOME = os.getcwd()
DATA_ROOT = '/content/drive/MyDrive/Dataset'
SEED = 42
# what we count as an image file
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

# dataset splits
EXPERIMENTS = ['EXP1_ALL', 'EXP2_OCCLUSION', 'EXP3_CLUSTER', 'EXP4_ILLUMINATION', 'EXP5_SCALE']

# yolov8 comes in 5 sizes - n(ano), s(mall), m(edium), l(arge), x(tra large). we train all of them
MODEL_SIZES = ['n', 's', 'm', 'l', 'x']

# type of run (for output dir)
TYPE_OF_RUN = 'YOLOv8'

CONFIG = {
    'epochs': 100,
    'batch_size': 16,
    'save_path': f'{HOME}/results',
}

In [ ]:
# the dataset already has YOLO_BOX folders with proper bbox labels, so no conversion needed
# this just makes sure each experiment has a data.yaml that ultralytics can load
def ensure_data_yaml(box_root):
  yaml_path = Path(box_root) / 'data.yaml'
  # if the yaml already exists we're good, leave it alone
  if yaml_path.exists():
    return str(yaml_path)
  # otherwise write a minimal yolo-style data.yaml pointing at the splits
  with open(yaml_path, 'w') as f:
    yaml.safe_dump({
      'path': str(box_root),
      'train': 'train/images',
      'val': 'valid/images',
      'test': 'test/images',
      'nc': 1, # only one class
      'names': ['orange'],
    }, f, sort_keys=False)
  return str(yaml_path)

In [ ]:
# how many image files live in a given split folder?
def count_images(split_dir):
  p = Path(split_dir)
  # if the folder doesn't exist, just say 0 instead of crashing
  if not p.exists():
    return 0
  # count files with image extensions
  return sum(1 for f in p.iterdir() if f.suffix.lower() in IMAGE_EXTS)

# total number of objects across all label files in a folder (each line = 1 object)
def count_objects(label_dir):
  p = Path(label_dir)
  if not p.exists():
    return 0
  total = 0
  # add up object counts across all label files
  for lbl in p.glob('*.txt'):
    with open(lbl) as f:
      # ignore blank lines
      total += sum(1 for line in f if line.strip())
  return total

# bundle up image + object counts for all 3 splits into one dict
def get_dataset_sizes(dataset_dir):
  sizes = {}
  # walk through each split and count its images + boxes
  for split in ['train', 'valid', 'test']:
    sizes[f'{split}_images'] = count_images(Path(dataset_dir) / split / 'images')
    sizes[f'{split}_oranges'] = count_objects(Path(dataset_dir) / split / 'labels')
  # roll up totals so the print summary is easy to eyeball
  sizes['total_images'] = sum(sizes[f'{s}_images'] for s in ['train', 'valid', 'test'])
  sizes['total_oranges'] = sum(sizes[f'{s}_oranges'] for s in ['train', 'valid', 'test'])
  return sizes

# how many objects are in a single label file?
def ground_truth_count(label_path):
  if not os.path.exists(label_path):
    return 0
  with open(label_path) as f:
    return sum(1 for line in f if line.strip())

In [ ]:
# train one yolov8 size on one experiment, save weights, and copy results to drive
def run_model(dataset, model_size, exp_name, trace=True):
  # yolov8 pretrained weights
  model_file = f'yolov8{model_size}.pt'
  # each run will have its own folder so we don't overwrite anything
  run_name = f'yolov8{model_size}_{exp_name}'

  # load pretrained checkpoint
  model = YOLO(model_file)
  # pass settings to ultralytics
  results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=CONFIG['epochs'],
    batch=CONFIG['batch_size'],
    save=True,
    save_period=10, # checkpoint every 10 epochs
    project=CONFIG['save_path'],
    name=run_name,
    exist_ok=True, # ok to reuse the folder if rerunning
    plots=True,
  )

  run_dir = os.path.join(CONFIG['save_path'], run_name)
  best_path = os.path.join(run_dir, 'weights', 'best.pt')

  # show the training curves + confusion matrix inline so we can make sure everything is ok
  if trace:
    for fname in ['results.png', 'confusion_matrix.png']:
      path = os.path.join(run_dir, fname)
      if os.path.exists(path):
        display(Image(filename=path, width=700))

  # copy whole run folder to drive
  drive_run_dir = os.path.join(DATA_ROOT, 'saved_models', TYPE_OF_RUN, run_name)
  shutil.copytree(run_dir, drive_run_dir, dirs_exist_ok=True)
  print(f'Saved run to: {drive_run_dir}')

  return model, results, run_dir, best_path

In [ ]:
# sweep confidence thresholds on val set and pick the one with lowest count error
def search_conf_threshold(best_path, dataset_location, thresholds=(0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9)):
  # load best weights from training
  model = YOLO(best_path)
  # tune on val
  val_img_dir = Path(dataset_location) / 'valid' / 'images'
  val_label_dir = Path(dataset_location) / 'valid' / 'labels'
  image_paths = sorted(p for p in val_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

  results = []
  # try each threshold and see which one predicts the most accurate orange counts
  for conf in thresholds:
    errors = []
    for img_path in image_paths:
      gt = ground_truth_count(val_label_dir / f'{img_path.stem}.txt')
      # run yolo at this confidence level
      pred = model.predict(source=str(img_path), conf=conf, verbose=False)[0]
      # if no detections, boxes is None
      pred_count = 0 if pred.boxes is None else len(pred.boxes)
      errors.append(pred_count - gt)
    # mae is the main metric we care about, rmse just for context (penalizes big mistakes more)
    errors = np.array(errors, dtype=float)
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    results.append({'conf': conf, 'mae': mae, 'rmse': rmse})
    print(f'conf={conf:.2f}  MAE={mae:.3f}  RMSE={rmse:.3f}')

  # pick threshold with lowest mae
  df = pd.DataFrame(results)
  best = df.loc[df['mae'].idxmin()]
  print(f"\nBest threshold: {best['conf']} (MAE={best['mae']:.3f})")
  return df, best['conf']

# run model on test set with chosen threshold and return all the metrics
def eval_model(best_path, dataset, conf=0.25):
  model = YOLO(best_path)
  # ultralytics calculates precision/recall/mAP for us already
  metrics = model.val(data=f'{dataset.location}/data.yaml', split='test', plots=True)
  rd = metrics.results_dict

  # pull out the box-level p/r and compute f1 manually
  p = rd['metrics/precision(B)']
  r = rd['metrics/recall(B)']
  f1 = (2 * p * r) / (p + r) if (p + r) else 0.0

  # we still want count error on test images, so loop one more time
  test_img_dir = Path(dataset.location) / 'test' / 'images'
  test_label_dir = Path(dataset.location) / 'test' / 'labels'
  image_paths = sorted(x for x in test_img_dir.iterdir() if x.suffix.lower() in IMAGE_EXTS)

  errors = []
  for img_path in image_paths:
    gt = ground_truth_count(test_label_dir / f'{img_path.stem}.txt')
    pred = model.predict(source=str(img_path), conf=conf, verbose=False)[0]
    pred_count = 0 if pred.boxes is None else len(pred.boxes)
    # positive = over-counted, negative = under-counted
    errors.append(pred_count - gt)
  errors = np.array(errors, dtype=float)

  # bundle everything into a dict so it slots into the results dataframe
  return {
    'box_precision': p,
    'box_recall': r,
    'box_f1': f1,
    'box_map50': rd['metrics/mAP50(B)'], # mAP at iou >= 0.5
    'box_map': rd['metrics/mAP50-95(B)'], # mean across iou thresholds 0.5 to 0.95
    'mae': np.mean(np.abs(errors)),
    'rmse': np.sqrt(np.mean(errors ** 2)),
  }

In [ ]:
# end-to-end pipeline for one experiment: train, tune conf, evaluate
def run_experiment(exp_name):
  # the bbox-format dataset
  box_root = os.path.join(DATA_ROOT, exp_name, 'YOLO_BOX')
  print(f'\n{exp_name}')
  print(f'Dataset: {box_root}')

  # make sure there's data.yaml for ultralytics
  ensure_data_yaml(box_root)
  # tiny shim so rest of the code can do dataset.location
  dataset = SimpleNamespace(location=box_root)
  # sanity-check sizes
  print(get_dataset_sizes(box_root))

  rows = []
  # train every yolov8 size so we can compare
  for ms in MODEL_SIZES:
    start = datetime.now()
    _, _, run_dir, best_path = run_model(dataset, model_size=ms, exp_name=exp_name)
    end = datetime.now()

    # tune conf on val, then evaluate on test
    _, best_conf = search_conf_threshold(best_path, box_root)
    metrics = eval_model(best_path, dataset, conf=best_conf)

    rows.append({
      'experiment': exp_name,
      'model': 'YOLOv8',
      'model_size': ms,
      'run_dir': run_dir,
      'best_path': best_path,
      'conf': best_conf,
      'train_time': end - start,
      **metrics,
    })

  return pd.DataFrame(rows)


# clear gpu memory
torch.cuda.empty_cache()

# loop through every experiment back to back
all_results = [run_experiment(exp) for exp in EXPERIMENTS]

results_df = pd.concat(all_results, ignore_index=True)

results_df.to_csv(f'{DATA_ROOT}/yolov8_results-{datetime.now():%Y%m%d-%H%M%S}.csv', index=False)
results_df

In [ ]:
# how many test images to draw per experiment
NUM_SAMPLES = 5

# loop through every experiment and visualize a few test predictions
for exp_name in EXPERIMENTS:
  # grab just the rows for this experiment
  subset = results_df[results_df['experiment'] == exp_name]
  if subset.empty:
    continue

  # pick best yolo size for this experiment (highest f1)
  # we might want to change this (depending on if we want f1 or mae to be our primary!!)
  row = subset.loc[subset['box_f1'].idxmax()]
  # rebuild + load weights for that run
  model = YOLO(row['best_path'])
  # use tuned conf threshold from eval
  conf = row['conf']

  box_root = os.path.join(DATA_ROOT, exp_name, 'YOLO_BOX')
  test_img_dir = Path(box_root) / 'test' / 'images'
  test_label_dir = Path(box_root) / 'test' / 'labels'
  image_paths = sorted(p for p in test_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

  # fixed seed so same images show up every time we re-run
  random.seed(SEED)
  sample_paths = random.sample(image_paths, min(NUM_SAMPLES, len(image_paths)))

  # one row per sample image
  fig, axes = plt.subplots(len(sample_paths), 1, figsize=(14, 6 * len(sample_paths)))
  if len(sample_paths) == 1:
    axes = [axes]

  for i, img_path in enumerate(sample_paths):
    # opencv loads as bgr by default, convert to rgb for matplotlib
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    label_path = test_label_dir / f'{img_path.stem}.txt'
    gt_count = ground_truth_count(label_path)

    # run the model on this image
    pred = model.predict(source=str(img_path), conf=conf, verbose=False)[0]
    overlay = img_rgb.copy()
    pred_count = 0 if pred.boxes is None else len(pred.boxes)
    if pred.boxes is not None:
      # draw (green) rectangle for every detection that survived the conf threshold
      for box in pred.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (105, 255, 99), 4)

    axes[i].imshow(overlay)
    axes[i].set_title(f'pred: {pred_count} / gt: {gt_count} (conf>={conf:.2f}, size={row["model_size"]})',
                      fontsize=14, fontweight='bold')
    axes[i].axis('off')

  plt.suptitle(f'Detection: {exp_name}', fontsize=16, fontweight='bold', y=1.0)
  plt.tight_layout()
  plt.show()